- Notebook modificado a partir de load_local_model.ipynb
- Carga imagenes del dataset galaxy10sdss, las pasa por el mmae-vit y almacena la representación fusionada.
    - Las imagenes solo tienen tres canales y no tienen espectro. Para ambos casos, se rellena con cero lo faltante.

- Almacena la representación fusionada en otro dataset
- Entrena un MLP con el dataset fusionado y registra resultados en mlflow


In [39]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import mlflow.pytorch
from galaxy_loader import GalaxyDataset
import numpy as np
import h5py
from astropy.visualization import (MinMaxInterval, LogStretch, SqrtStretch,
                                   AsinhStretch, PowerStretch, ImageNormalize,
                                   ZScaleInterval, LinearStretch)
import seaborn as sns
import matplotlib.patches as patches
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
import pandas as pd

# Cargar modelo mmae-vit

In [40]:
# Define las rutas
model_uri = "./model_artifacts/best_model"


In [41]:
# ——————————————————————————————————————————————
# 1) Instantiate & load your new model
# ——————————————————————————————————————————————
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Step 2: Load model using MLflow
# Lo de map_location no se ocupa si tienes gpu 
mmae = mlflow.pytorch.load_model(model_uri,map_location='cpu')
print(mmae)

2026/10/25 00:11:47 WARNING mlflow.pytorch: Stored model version '2.14.1+cu132' does not match installed PyTorch version '2.9.1'


MMAE_ViT(
  (img_embed): PatchEmbed(
    (proj): Conv2d(5, 256, kernel_size=(8, 8), stride=(8, 8))
  )
  (spec_embed): SpectrumPatchEmbed(
    (linear): Linear(in_features=8, out_features=256, bias=True)
    (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (img_encoder): TransformerEncoder1D(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-3): 4 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
          )
          (linear1): Linear(in_features=256, out_features=1024, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=1024, out_features=256, bias=True)
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dr

# Modelo de clasificación

### Por el momento, MLP. 

In [42]:
class NN_galaxias(nn.Module):
    def __init__(self, input_features=512, hidden_features=256, num_classes=10):
        super().__init__()

        self.feature_layer = nn.Linear(input_features, hidden_features)  # 512 -> 256
        self.hidden_layer = nn.Linear(hidden_features, hidden_features // 2)  # 256 -> 128
        self.hidden_layer2 = nn.Linear(hidden_features // 2, hidden_features // 4)  # 128 -> 64
        self.classification_head = nn.Linear(hidden_features // 4, num_classes)  # 64 -> 10

    def forward(self, features):
        x = F.relu(self.feature_layer(features))
        x = F.relu(self.hidden_layer(x))
        x = F.relu(self.hidden_layer2(x))
        return self.classification_head(x)

# Aplicamos MMAE-vit a todo el dataset de clasificación de galaxias

## Carga datos

### Lo pueden descomentar la primera vez para que descargue los datos y los guarde en su computadora

In [43]:
# # import everything we need first
# from tensorflow.keras import utils
# import numpy as np
# from sklearn.model_selection import train_test_split
# import pylab as plt

# from astroNN.models import Galaxy10CNN
# from astroNN.datasets import load_galaxy10sdss
# from astroNN.datasets.galaxy10sdss import galaxy10cls_lookup, galaxy10_confusion

# # To load images and labels (will download automatically at the first time)
# # First time downloading location will be ~/.astroNN/datasets/
# images, labels = load_galaxy10sdss()

# # To convert the labels to categorical 10 classes
# labels = utils.to_categorical(labels, 10)

# # To convert to desirable type
# labels = labels.astype(np.float32)
# images = images.astype(np.float32)

# # Split into train/validation/test sets with proportions 70/15/15
# all_indices = np.arange(labels.shape[0])
# class_ids = labels.argmax(axis=1)
# train_val_idx, test_idx = train_test_split(
#     all_indices,
#     test_size=0.15,
#     random_state=42,
#     stratify=class_ids,
# )
# train_idx, val_idx = train_test_split(
#     train_val_idx,
#     test_size=0.15 / 0.85,
#     random_state=42,
#     stratify=class_ids[train_val_idx],
# )
# train_images, train_labels = images[train_idx], labels[train_idx]
# val_images, val_labels = images[val_idx], labels[val_idx]
# test_images, test_labels = images[test_idx], labels[test_idx]


## Aplicar mmae-vit y extraer fused_representation

In [44]:
def extract_mmae_features(images, labels, train_idx, val_idx, test_idx, model, output_path="galaxy10_mmae_features.npz", batch_size=64):
    images = torch.as_tensor(images, dtype=torch.float32)
    if images.ndim != 4 or images.shape[-1] != 3:
        raise ValueError(f"Se esperaban imágenes GRI con forma (N, H, W, 3); se recibió {tuple(images.shape)}")
    if images.shape[0] != len(labels):
        raise ValueError("El número de imágenes no coincide con el número de etiquetas.")
    if len(images) == 0 or batch_size <= 0:
        raise ValueError("El conjunto de imágenes debe ser no vacío y batch_size debe ser positivo.")

    image_scale = 255.0 if images.max() > 1.0 else 1.0
    image_size = model.img_size

    label_tensor = torch.as_tensor(labels)
    if label_tensor.ndim == 2 and label_tensor.shape[1] == 10:
        class_labels = label_tensor.argmax(dim=1).to(dtype=torch.long).cpu().numpy()
    elif label_tensor.ndim == 1:
        class_labels = label_tensor.to(dtype=torch.long).cpu().numpy()
    else:
        raise ValueError(f"Las etiquetas deben tener forma (N,) o (N, 10); se recibió {tuple(label_tensor.shape)}")

    extraction_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(extraction_device)
    model.eval()
    feature_batches = []
    with torch.no_grad():
        for start in range(0, len(images), batch_size):
            rgb_batch = images[start : start + batch_size].permute(0, 3, 1, 2)
            rgb_batch = rgb_batch / image_scale
            rgb_batch = F.interpolate(
                rgb_batch,
                size=(image_size, image_size),
                mode="bilinear",
                align_corners=False,
            )
            # Rellenar en ceros los otros dos canales
            missing_image_channels = torch.zeros(
                rgb_batch.shape[0], 2, image_size, image_size, dtype=rgb_batch.dtype
            )
            image_batch = torch.cat((rgb_batch, missing_image_channels), dim=1).to(extraction_device)
            # Espectro vacio
            spectrum_batch = torch.zeros(
                image_batch.shape[0], 1, model.spectrum_len, device=extraction_device
            )
            outputs = model(
                image_batch,
                spec=spectrum_batch,
                img_mask_ratio=0.0,
                spec_mask_ratio=0.0,
                use_spec=True,
            )
            fused_batch = outputs["fused_representation"]
            if fused_batch.ndim != 2 or fused_batch.shape[1] != 512:
                raise ValueError(
                    f"Se esperaban features fusionadas con forma (N, 512); se recibió {tuple(fused_batch.shape)}"
                )
            feature_batches.append(fused_batch.cpu())

    fused_features = torch.cat(feature_batches).numpy().astype(np.float32, copy=False)
    if len(fused_features) != len(class_labels):
        raise ValueError("El número de features extraídas no coincide con el número de etiquetas.")

    np.savez_compressed(
        output_path,
        features=fused_features,
        labels=class_labels,
        train_idx=np.asarray(train_idx),
        val_idx=np.asarray(val_idx),
        test_idx=np.asarray(test_idx),
    )
    print(f"Dataset guardado en {output_path}: features {fused_features.shape}, etiquetas {class_labels.shape}")
    return fused_features, class_labels

In [45]:
# fused_features, class_labels = extract_mmae_features(
#     images=images,
#     labels=labels,
#     train_idx=train_idx,
#     val_idx=val_idx,
#     test_idx=test_idx,
#     model=mmae,
#     output_path="galaxy10_mmae_features.npz",
#     batch_size=64,
# )

# Entrenamiento clasificador

## Cargar datos de galaxy10_mmae_features.npz

In [46]:
classification_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

feature_dataset_path = "galaxy10_mmae_features.npz"
with np.load(feature_dataset_path) as feature_dataset:
    required_keys = {"features", "labels", "train_idx", "val_idx", "test_idx"}
    missing_keys = required_keys.difference(feature_dataset.files)
    if missing_keys:
        raise ValueError(f"El archivo {feature_dataset_path} no contiene: {sorted(missing_keys)}")

    all_features = torch.from_numpy(feature_dataset["features"].copy()).to(dtype=torch.float32)
    all_targets = torch.from_numpy(feature_dataset["labels"].copy()).to(dtype=torch.long)
    train_indices = torch.from_numpy(feature_dataset["train_idx"].copy()).to(dtype=torch.long)
    val_indices = torch.from_numpy(feature_dataset["val_idx"].copy()).to(dtype=torch.long)
    test_indices = torch.from_numpy(feature_dataset["test_idx"].copy()).to(dtype=torch.long)

if all_features.ndim != 2 or all_features.shape[1] != 512:
    raise ValueError(f"Se esperaban features con forma (N, 512), se recibió {tuple(all_features.shape)}")
if all_targets.ndim != 1 or all_targets.shape[0] != all_features.shape[0]:
    raise ValueError("Las etiquetas deben tener forma (N,) y corresponder a las features.")
if train_indices.numel() == 0 or val_indices.numel() == 0 or test_indices.numel() == 0:
    raise ValueError("Los índices de entrenamiento, validación y prueba no pueden estar vacíos.")
if torch.any((train_indices < 0) | (train_indices >= len(all_features))) or torch.any((val_indices < 0) | (val_indices >= len(all_features))) or torch.any((test_indices < 0) | (test_indices >= len(all_features))):
    raise ValueError("El archivo contiene índices fuera del rango del dataset de features.")
if len(torch.unique(torch.cat((train_indices, val_indices, test_indices)))) != len(all_features):
    raise ValueError("Los índices deben particionar todas las features sin traslapes.")

train_features = all_features[train_indices]
val_features = all_features[val_indices]
test_features = all_features[test_indices]

if train_features.ndim != 2 or train_features.shape[1] != 512:
    raise ValueError(f"Se esperaban train_images con forma (N, 512), se recibió {tuple(train_features.shape)}")
if test_features.ndim != 2 or test_features.shape[1] != 512:
    raise ValueError(f"Se esperaban test_images con forma (N, 512), se recibió {tuple(test_features.shape)}")
train_targets = all_targets[train_indices]
val_targets = all_targets[val_indices]
test_targets = all_targets[test_indices]
if train_features.shape[0] != train_targets.shape[0]:
    raise ValueError("El número de features de entrenamiento no coincide con el de etiquetas.")
if test_features.shape[0] != test_targets.shape[0]:
    raise ValueError("El número de features de prueba no coincide con el de etiquetas.")
if val_features.shape[0] != val_targets.shape[0]:
    raise ValueError("El número de features de validación no coincide con el de etiquetas.")
if torch.any((train_targets < 0) | (train_targets >= 10)) or torch.any((val_targets < 0) | (val_targets >= 10)) or torch.any((test_targets < 0) | (test_targets >= 10)):
    raise ValueError("Las etiquetas deben ser índices de clase entre 0 y 9.")


In [47]:
print(train_features.shape)
print(val_features.shape)
print(test_features.shape)

torch.Size([15249, 512])
torch.Size([3268, 512])
torch.Size([3268, 512])


## Entrenar y registrar resultados en ML_flow

In [48]:
def carga_clasificador(dir_checkpoint="model_artifacts/galaxy10/checkpoints", nombre_checkpoint=""):
    checkpoint_dir = Path(dir_checkpoint)
    checkpoint_name = f"nn_galaxias_{nombre_checkpoint}.pth"
    classifier_checkpoint = checkpoint_dir / checkpoint_name
    classifier_checkpoint.parent.mkdir(parents=True, exist_ok=True)
    classifier = NN_galaxias(**classifier_config).to(classification_device)
    if classifier_checkpoint.is_file():
        checkpoint = torch.load(
            classifier_checkpoint,
            map_location=classification_device,
            weights_only=True,
        )
        if checkpoint.get("model_config") != classifier_config:
            # raise ValueError(
            #     f"La configuración guardada en {classifier_checkpoint} no coincide con NN_galaxias."
            # )
            print(f"La configuración guardada en {classifier_checkpoint} no coincide con NN_galaxias.")
        else:
            classifier.load_state_dict(checkpoint["state_dict"])
            print(f"Clasificador cargado desde {classifier_checkpoint}")
    else:
        print("No hay checkpoint de NN_galaxias; se inicializa un modelo nuevo.")
    return classifier,classifier_checkpoint


In [49]:
# Entrenamiento del clasificador con features de 512 dimensiones
batch_size = 256
num_epochs = 20
learning_rate = 0.001
hidden_features= 256

train_loader = DataLoader(
    torch.utils.data.TensorDataset(train_features, train_targets),
    batch_size=batch_size,
    shuffle=True,
)
val_loader = DataLoader(
    torch.utils.data.TensorDataset(val_features, val_targets),
    batch_size=batch_size,
    shuffle=False,
)
test_loader = DataLoader(
    torch.utils.data.TensorDataset(test_features, test_targets),
    batch_size=batch_size,
    shuffle=False,
)
classifier_config = {
    "input_features": 512,
    "hidden_features": 256,
    "num_classes": 10,
}
nombre_run="ejemplo"
# Cargar checkpoint
classifier,classifier_checkpoint=carga_clasificador(nombre_checkpoint=nombre_run)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(classifier.parameters(), lr=learning_rate, momentum=0.9)

# por si se quedo abierta una antes xd
mlflow.end_run()

# MLFLOW
mlflow.set_tracking_uri(
    "sqlite:////Users/leopoldomorante/mlflow_sample/mlflow_sample/mlflow.db"
)
# Reinicia MLflow UI usando la misma base: mlflow ui --backend-store-uri sqlite:////Users/leopoldomorante/mlflow_sample/mlflow_sample/mlflow.db

experiment_name = "galaxy10-classifier"
mlflow.set_experiment(experiment_name)

with mlflow.start_run(run_name=nombre_run) as active_run:
    mlflow.log_params(
        {
            "input_features": 512,
            "hidden_features": hidden_features,
            "num_classes": 10,
            "optimizer": "SGD",
            "learning_rate": learning_rate,
            "momentum": 0.9,
            "batch_size": batch_size,
            "num_epochs": num_epochs,
            "loss": "CrossEntropyLoss",
        }
    )

    for epoch in range(num_epochs):
        classifier.train()
        train_loss_sum = 0.0
        train_correct = 0
        for batch_features, batch_targets in train_loader:
            batch_features = batch_features.to(classification_device)
            batch_targets = batch_targets.to(classification_device)

            optimizer.zero_grad()
            logits = classifier(batch_features)
            loss = criterion(logits, batch_targets)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * batch_features.size(0)
            train_correct += (logits.argmax(dim=1) == batch_targets).sum().item()

        classifier.eval()
        val_loss_sum = 0.0
        val_correct = 0
        with torch.no_grad():
            for batch_features, batch_targets in val_loader:
                batch_features = batch_features.to(classification_device)
                batch_targets = batch_targets.to(classification_device)
                logits = classifier(batch_features)
                val_loss_sum += criterion(logits, batch_targets).item() * batch_features.size(0)
                val_correct += (logits.argmax(dim=1) == batch_targets).sum().item()

        train_loss = train_loss_sum / len(train_targets)
        train_accuracy = train_correct / len(train_targets)
        val_loss = val_loss_sum / len(val_targets)
        val_accuracy = val_correct / len(val_targets)
        mlflow.log_metrics(
            {
                "train_loss": train_loss,
                "train_accuracy": train_accuracy,
                "val_loss": val_loss,
                "val_accuracy": val_accuracy,
            },
            step=epoch + 1,
        )
        print(
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"train loss: {train_loss:.4f} | "
            f"train accuracy: {train_accuracy:.4f} | "
            f"val loss: {val_loss:.4f} | "
            f"val accuracy: {val_accuracy:.4f}"
        )



    torch.save(
        {
            "model_config": classifier_config,
            "state_dict": {
                name: tensor.detach().cpu()
                for name, tensor in classifier.state_dict().items()
            },
        },
        classifier_checkpoint,
    )
    print(f"Clasificador guardado en {classifier_checkpoint}")

    mlflow.pytorch.log_model(
        pytorch_model=classifier,
        artifact_path="classifier",
        input_example=train_features[:2].cpu().numpy(),
    )
    run_id = active_run.info.run_id

print(f"Experimento: {experiment_name} | run_id: {run_id}")
print(f"Modelo guardado en: runs:/{run_id}/classifier")



mlflow.end_run()

Clasificador cargado desde model_artifacts/galaxy10/checkpoints/nn_galaxias_ejemplo.pth
Epoch 1/20 | train loss: 1.6276 | train accuracy: 0.3707 | val loss: 1.6330 | val accuracy: 0.3727
Epoch 2/20 | train loss: 1.6233 | train accuracy: 0.3699 | val loss: 1.6296 | val accuracy: 0.3767
Epoch 3/20 | train loss: 1.6192 | train accuracy: 0.3747 | val loss: 1.6288 | val accuracy: 0.3840
Epoch 4/20 | train loss: 1.6154 | train accuracy: 0.3790 | val loss: 1.6211 | val accuracy: 0.3758
Epoch 5/20 | train loss: 1.6111 | train accuracy: 0.3764 | val loss: 1.6175 | val accuracy: 0.3822
Epoch 6/20 | train loss: 1.6070 | train accuracy: 0.3755 | val loss: 1.6120 | val accuracy: 0.3785
Epoch 7/20 | train loss: 1.6017 | train accuracy: 0.3813 | val loss: 1.6068 | val accuracy: 0.3791
Epoch 8/20 | train loss: 1.5979 | train accuracy: 0.3810 | val loss: 1.6086 | val accuracy: 0.3767
Epoch 9/20 | train loss: 1.5923 | train accuracy: 0.3798 | val loss: 1.5965 | val accuracy: 0.3807
Epoch 10/20 | train l

2026/10/25 00:11:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 19/20 | train loss: 1.5319 | train accuracy: 0.3931 | val loss: 1.5361 | val accuracy: 0.3972
Epoch 20/20 | train loss: 1.5257 | train accuracy: 0.3948 | val loss: 1.5304 | val accuracy: 0.3929
Clasificador guardado en model_artifacts/galaxy10/checkpoints/nn_galaxias_ejemplo.pth
Experimento: galaxy10-classifier | run_id: cc7a65f923dd436b871a97030438279a
Modelo guardado en: runs:/cc7a65f923dd436b871a97030438279a/classifier


# Evaluar un modelo guardado en conjunto de TEST

In [50]:
# nombre_checkpoint="ejemplo"
# classifier,_=carga_clasificador(nombre_checkpoint=nombre_checkpoint)

# mlflow.set_tracking_uri(
#     "sqlite:////Users/leopoldomorante/mlflow_sample/mlflow_sample/mlflow.db"
# )
# experiment_name = "galaxy10-classifier"
# mlflow.set_experiment(experiment_name)

# with mlflow.start_run(run_name=f"test_{nombre_run}") as active_run:
   
#     classifier.eval()
#     test_loss_sum = 0.0
#     test_correct = 0
#     with torch.no_grad():
#         for batch_features, batch_targets in test_loader:
#             batch_features = batch_features.to(classification_device)
#             batch_targets = batch_targets.to(classification_device)
#             logits = classifier(batch_features)
#             test_loss_sum += criterion(logits, batch_targets).item() * batch_features.size(0)
#             test_correct += (logits.argmax(dim=1) == batch_targets).sum().item()

#     test_loss = test_loss_sum / len(test_targets)
#     test_accuracy = test_correct / len(test_targets)
#     mlflow.log_metrics({"test_loss": test_loss, "test_accuracy": test_accuracy})
#     print(f"Evaluación final en test | loss: {test_loss:.4f} | accuracy: {test_accuracy:.4f}")
    
    
# mlflow.end_run()
